# 🏅 Olympic Dataset — EDA & Data Cleaning

**Dataset:** 120 Years of Olympic History (Kaggle)  
**Goal:** Understand the dataset structure, handle missing values, and prepare clean data for analysis.

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
%matplotlib inline

print('Libraries loaded successfully ✅')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../data/athlete_events.csv')

print(f'Shape: {df.shape}')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
df.head()

## 3. Dataset Overview

In [ ]:
# Column info
df.info()

In [ ]:
# Basic statistics
df.describe()

In [ ]:
# Unique values per column
for col in df.columns:
    print(f'{col}: {df[col].nunique()} unique values')

## 4. Missing Value Analysis

In [ ]:
# Missing value counts and percentages
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
})
missing = missing[missing['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print(missing)

In [ ]:
# Visualize missing values
plt.figure(figsize=(8, 4))
sns.barplot(x=missing.index, y=missing['Missing %'], palette='Blues_r')
plt.title('Missing Value Percentage by Column', fontsize=14)
plt.ylabel('Missing %')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('../visuals/missing_values.png', dpi=150)
plt.show()

## 5. Data Cleaning

In [ ]:
# Medal: NaN means no medal → fill with 'None'
df['Medal'] = df['Medal'].fillna('None')

# Age, Height, Weight: fill with median per sport
for col in ['Age', 'Height', 'Weight']:
    df[col] = df.groupby('Sport')[col].transform(lambda x: x.fillna(x.median()))

# Check remaining missing values
print('Remaining missing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Create medal indicator columns
df['Gold']   = (df['Medal'] == 'Gold').astype(int)
df['Silver'] = (df['Medal'] == 'Silver').astype(int)
df['Bronze'] = (df['Medal'] == 'Bronze').astype(int)
df['Total']  = df['Gold'] + df['Silver'] + df['Bronze']

print('Medal columns created ✅')
df[['Medal', 'Gold', 'Silver', 'Bronze', 'Total']].head(10)

## 6. General Exploration

In [ ]:
# Participation by year
yearly = df.groupby('Year')['ID'].nunique().reset_index()
yearly.columns = ['Year', 'Athletes']

plt.figure(figsize=(12, 4))
sns.lineplot(data=yearly, x='Year', y='Athletes', marker='o', color='steelblue')
plt.title('Number of Unique Athletes per Year', fontsize=14)
plt.ylabel('Athletes')
plt.tight_layout()
plt.savefig('../visuals/athletes_per_year.png', dpi=150)
plt.show()

In [ ]:
# Top 10 countries by total medals
top_countries = (df.groupby('Team')['Total'].sum()
                 .sort_values(ascending=False)
                 .head(10)
                 .reset_index())

plt.figure(figsize=(10, 5))
sns.barplot(data=top_countries, x='Total', y='Team', palette='YlOrRd_r')
plt.title('Top 10 Countries by Total Medals (All Time)', fontsize=14)
plt.tight_layout()
plt.savefig('../visuals/top_countries.png', dpi=150)
plt.show()

In [ ]:
# Age distribution of all athletes
plt.figure(figsize=(10, 4))
sns.histplot(df['Age'].dropna(), bins=40, color='steelblue', kde=True)
plt.title('Age Distribution of Olympic Athletes', fontsize=14)
plt.xlabel('Age')
plt.tight_layout()
plt.savefig('../visuals/age_distribution.png', dpi=150)
plt.show()

print(f"Mean age: {df['Age'].mean():.1f} | Median: {df['Age'].median():.1f}")

In [ ]:
# Gender distribution over years
gender = df.groupby(['Year', 'Sex'])['ID'].nunique().reset_index()
gender.columns = ['Year', 'Sex', 'Count']

plt.figure(figsize=(12, 4))
sns.lineplot(data=gender, x='Year', y='Count', hue='Sex',
             palette={'M': 'steelblue', 'F': 'salmon'}, marker='o')
plt.title('Male vs Female Athletes Over the Years', fontsize=14)
plt.tight_layout()
plt.savefig('../visuals/gender_over_years.png', dpi=150)
plt.show()

## 7. Save Cleaned Dataset

In [ ]:
df.to_csv('../data/athlete_events_clean.csv', index=False)
print(f'Cleaned dataset saved ✅ — Shape: {df.shape}')

---
## ✅ Summary

| Step | Action |
|------|--------|
| Missing Medal | Filled with 'None' |
| Missing Age/Height/Weight | Filled with sport-level median |
| Medal columns | Gold, Silver, Bronze, Total created |
| Output | `athlete_events_clean.csv` saved |

**Next:** `02_turkey_analysis.ipynb` → Turkey-specific deep dive